# ESC-50 project setup
Choose **Runtime → Change runtime type → T4 GPU → Save**, then run the cells in order.

In [ ]:
import subprocess

gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
).strip()
print(gpu)
assert "T4" in gpu.split(), "Select T4 GPU in Runtime → Change runtime type."


In [ ]:
from pathlib import Path
import os
import re

repo_url = "https://github.com/OWNER/REPO.git"  # Replace with your clone URL.
private_repo = False  # Set to True for a private repository.

if "OWNER/REPO" in repo_url or not re.fullmatch(
    r"https://github[.]com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+[.]git", repo_url
):
    raise ValueError("Enter your GitHub HTTPS clone URL ending in .git.")

project_dir = Path("/content/esc50-wav2vec2")
if project_dir.exists():
    existing_url = subprocess.check_output(
        ["git", "-C", str(project_dir), "remote", "get-url", "origin"], text=True
    ).strip()
    if existing_url != repo_url:
        raise ValueError(f"{project_dir} already contains a different repository.")
    print("Using existing clone.")
else:
    clone_env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    if private_repo:
        import base64
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
        auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        clone_env.update(
            GIT_CONFIG_COUNT="1",
            GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
            GIT_CONFIG_VALUE_0=f"Authorization: Basic {auth}",
        )
    try:
        subprocess.run(["git", "clone", repo_url, str(project_dir)], env=clone_env, check=True)
    finally:
        if private_repo:
            del token, auth
        del clone_env


In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt"],
    cwd=project_dir,
    check=True,
)
subprocess.run(["bash", "run.sh", "42"], cwd=project_dir, check=True)
